# Lab : Construire un Agent ReAct avec ses propres Tools

---

## Objectifs pédagogiques

À la fin de ce lab, vous serez capable de :

1. Écrire des **tools from scratch** (fonction simple, puis avec schéma Pydantic)
2. Comprendre comment un LLM "décide" d'appeler un tool (`bind_tools`, `tool_calls`)
3. Implémenter **manuellement** la boucle ReAct (Reason → Act → Observe)
4. Construire un **agent ReAct avec LangGraph** (StateGraph, nœuds, arêtes conditionnelles)
5. Ajouter de la **mémoire** à l'agent (checkpointer)
6. Brancher un **tool de recherche web réel** (Tavily) pour aller chercher des
   informations à jour (taux de change actuels, recherche internet générale)

## Les bibliothèques du lab

| Bibliothèque | Rôle |
|---|---|
| **langchain** | Framework de haut niveau : modèles de chat, tools, agents préconstruits |
| **langchain-core** | Briques de base : classes `tool`, messages (`HumanMessage`, `ToolMessage`...) |
| **langgraph** | Orchestration : modéliser l'agent comme un **graphe** (nœuds + arêtes) avec état et mémoire |
| **pydantic** | Validation de données : définir et valider les schémas d'arguments des tools |
| **python-dotenv** | Charger les variables d'environnement (clés API) depuis un fichier `.env` |
| **langchain-tavily** | Tool de recherche web préconstruit, connecté à l'API de recherche **Tavily** |

## Ce lab est indépendant du fournisseur de LLM

Ce lab **ne suppose pas** que vous utilisez OpenAI. Vous pouvez utiliser
**n'importe quel fournisseur de modèle de chat compatible "tool calling"** :
OpenAI, Anthropic (Claude), Google (Gemini), Mistral, Groq, Ollama en local, etc.

Vous choisirez votre fournisseur dans la **Partie 0**, et tout le reste du lab
fonctionnera sans modification (c'est justement l'intérêt de `init_chat_model`).

---
# Partie 0 — Installation et configuration

Exécutez la cellule d'installation, puis **choisissez votre fournisseur de LLM**
(celui pour lequel vous avez une clé API : OpenAI, Anthropic, Google, Mistral,
Groq, ou un modèle local via Ollama...).

Vous aurez également besoin d'une clé **Tavily** (gratuite) pour les tools de
recherche web de la Partie 1 : créez un compte sur https://tavily.com pour
obtenir votre `TAVILY_API_KEY`.

In [7]:
# À exécuter une seule fois, puis redémarrer le kernel si nécessaire
# Le package du fournisseur LLM (ex: langchain-anthropic) dépend de votre choix,
# installez-le en plus si besoin (voir tableau ci-dessous).
! pip install -q langchain langgraph python-dotenv grandalf langchain-tavily


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\vargnomad\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Documentation — Choisir son fournisseur de LLM

**`langchain.chat_models.init_chat_model(model, **kwargs)`**
Fabrique **universelle** de modèles de chat. Au lieu d'importer une classe par
fournisseur (`ChatOpenAI`, `ChatAnthropic`...), on passe une chaîne
`"fournisseur:nom_du_modele"`. C'est ce mécanisme qui rend ce lab **agnostique**
du fournisseur choisi.

| Fournisseur | Package à installer | Chaîne `init_chat_model` | Variable d'env. clé API |
|---|---|---|---|
| OpenAI | `langchain-openai` | `"openai:gpt-4o-mini"` | `OPENAI_API_KEY` |
| Anthropic | `langchain-anthropic` | `"anthropic:claude-sonnet-4-5"` | `ANTHROPIC_API_KEY` |
| Google Gemini | `langchain-google-genai` | `"google_genai:gemini-2.5-flash"` | `GOOGLE_API_KEY` |
| Mistral | `langchain-mistralai` | `"mistralai:mistral-large-latest"` | `MISTRAL_API_KEY` |
| Groq | `langchain-groq` | `"groq:llama-3.3-70b-versatile"` | `GROQ_API_KEY` |
| Ollama (local) | `langchain-ollama` | `"ollama:llama3.1"` | *(aucune clé, serveur local)* |

**Le modèle choisi doit supporter le "tool calling"** (appel de fonctions) —
c'est le cas de tous les modèles listés ci-dessus dans leurs versions récentes.

Créez un fichier **`.env`** à côté de ce notebook et renseignez-y **uniquement**
la ou les clés correspondant à votre choix, par exemple :

```
ANTHROPIC_API_KEY=sk-ant-...
TAVILY_API_KEY=tvly-...
```

**`dotenv.load_dotenv()`** lit ce fichier et injecte les variables dans
`os.environ` : vous n'écrivez donc **jamais** vos clés dans le code.

### À vous de jouer

1. Installez le package de **votre** fournisseur (ex : `! pip install -q langchain-anthropic`)
2. Renseignez `MODEL_ID` ci-dessous avec la chaîne correspondant à votre choix
3. Complétez votre fichier `.env`

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # charge les clés API depuis le fichier .env

# Google Gemini — nécessite GOOGLE_API_KEY dans le fichier .env
# et le package : pip install langchain-google-genai
MODEL_ID = "gemini-3.5-flash"

llm = init_chat_model(MODEL_ID, temperature=0)
print(llm.invoke("Dis bonjour en une phrase").content)

---
# Partie 1 — Écrire des tools from scratch

Un **tool** est une fonction Python que le LLM peut décider d'appeler.
Le LLM ne voit **jamais** le code du tool. Il ne voit que 3 choses :

1. son **nom** (`name`)
2. sa **description** (`description`) → c'est ce qui lui permet de choisir le bon tool !
3. son **schéma d'arguments** (`args`) → noms, types, descriptions des paramètres

 Retenez : *un tool bien décrit = un agent qui choisit bien.*

## Exercice 1.1 — Tools avec le décorateur `@tool`

### Documentation — `langchain_core.tools.tool`

```python
from langchain_core.tools import tool
```

Le décorateur `@tool` transforme une simple fonction Python en objet **`StructuredTool`**
utilisable par un LLM. Il construit automatiquement :

| Attribut du tool | Source dans votre code |
|---|---|
| `name` | le nom de la fonction (`additionner`) |
| `description` | la **docstring** (obligatoire !) |
| `args` | les **annotations de types** des paramètres (`a: float`) |

Un tool s'exécute avec **`.invoke(dict_arguments)`** :

```python
additionner.invoke({"a": 3, "b": 4})   # → 7.0
```

**Règle d'or pour les erreurs** : dans un tool, on préfère **retourner un message
d'erreur en texte** plutôt que lever une exception. Ainsi l'agent peut "lire" l'erreur
comme une observation et se corriger tout seul.

### À vous de jouer

Créez 3 tools de calcul : `additionner`, `multiplier`, `diviser`.

In [ ]:
from langchain_core.tools import tool

@tool
def additionner(a: float, b: float) -> float:
    """Additionne deux nombres a et b."""
    return a + b

@tool
def multiplier(a: float, b: float) -> float:
    """Multiplie deux nombres a et b."""
    return a * b

@tool
def diviser(a: float, b: float) -> str:
    """Divise le nombre a par le nombre b. Retourne une erreur si b vaut zéro."""
    if b == 0:
        return "Erreur : division par zéro impossible."
    return str(a / b)


In [ ]:
# Zone de test — Exercice 1.1
print("name        :", additionner.name)
print("description :", additionner.description)
print("args        :", additionner.args)
print()
print("additionner(3, 4)  =", additionner.invoke({"a": 3, "b": 4}))
print("multiplier(6, 7)   =", multiplier.invoke({"a": 6, "b": 7}))
print("diviser(10, 3)     =", diviser.invoke({"a": 10, "b": 3}))
print("diviser(10, 0)     =", diviser.invoke({"a": 10, "b": 0}))


## Exercice 1.2 — Tool avec schéma Pydantic (validation des arguments)

### Documentation — `pydantic.BaseModel` et `Field`

```python
from pydantic import BaseModel, Field
```

**Pydantic** est LA bibliothèque de validation de données en Python. Un schéma se
déclare comme une classe héritant de **`BaseModel`** ; chaque attribut est un champ typé.

**`Field(...)`** enrichit un champ avec :
- `description=` → texte lu par le **LLM** pour comprendre le paramètre
- des **contraintes de validation** : `gt=0` (strictement supérieur à 0), `ge`, `lt`, `le`, `max_length`...

Si le LLM envoie un argument invalide (ex : `montant = -5`), Pydantic lève une
**`ValidationError`** *avant même* que votre code ne s'exécute : c'est un garde-fou gratuit.

### Documentation — `@tool(args_schema=...)`

En passant `args_schema=MonSchema` au décorateur, on remplace le schéma déduit des
annotations par notre schéma Pydantic, plus riche et plus strict :

```python
@tool(args_schema=ConversionInput)
def convertir_devise(montant: float, ...) -> str:
    ...
```

Les noms des paramètres de la fonction doivent correspondre aux champs du schéma.

> Ce tool utilise des taux **codés en dur** (figés dans le temps). Vous verrez
> en 1.4 comment obtenir un **taux réel et à jour** grâce à la recherche web.

### À vous de jouer

Créez un tool `convertir_devise` qui convertit un montant entre **MAD, EUR et USD**
(taux codés en dur).

In [ ]:
from pydantic import BaseModel, Field

TAUX = {("MAD", "EUR"): 0.093, ("EUR", "MAD"): 10.75,
        ("MAD", "USD"): 0.10,  ("USD", "MAD"): 10.0,
        ("EUR", "USD"): 1.08,  ("USD", "EUR"): 0.93}

class ConversionInput(BaseModel):
    montant: float = Field(description="Le montant à convertir", gt=0)
    devise_source: str = Field(description="Code devise source : MAD, EUR ou USD")
    devise_cible: str = Field(description="Code devise cible : MAD, EUR ou USD")

@tool(args_schema=ConversionInput)
def convertir_devise(montant: float, devise_source: str, devise_cible: str) -> str:
    """Convertit un montant d'une devise vers une autre (MAD, EUR, USD) avec un taux fixe."""
    devise_source = devise_source.upper()
    devise_cible = devise_cible.upper()
    if devise_source == devise_cible:
        return f"{montant} {devise_source} = {montant} {devise_cible} (même devise)"
    paire = (devise_source, devise_cible)
    if paire not in TAUX:
        return f"Erreur : conversion de {devise_source} vers {devise_cible} non supportée."
    resultat = round(montant * TAUX[paire], 2)
    return f"{montant} {devise_source} = {resultat} {devise_cible}"


In [ ]:
# Zone de test — Exercice 1.2
print(convertir_devise.invoke({"montant": 150, "devise_source": "mad", "devise_cible": "eur"}))
print(convertir_devise.invoke({"montant": 100, "devise_source": "EUR", "devise_cible": "USD"}))
print(convertir_devise.invoke({"montant": 500, "devise_source": "MAD", "devise_cible": "MAD"}))

# Test avec montant négatif : Pydantic lève une ValidationError avant l'exécution
try:
    print(convertir_devise.invoke({"montant": -5, "devise_source": "EUR", "devise_cible": "MAD"}))
except Exception as e:
    print(f"Erreur de validation Pydantic interceptée : {type(e).__name__}")


## Exercice 1.3 — Un tool de recherche dans un catalogue

### Documentation — logique métier dans un tool

Rien n'empêche un tool de contenir de la logique un peu plus riche : boucles,
recherche partielle, dictionnaires de données... Tant que la fonction reste
**pure** (mêmes arguments → même résultat, pas d'effet de bord caché) et que sa
**docstring** décrit bien son usage, `@tool` suffit — pas besoin d'une classe.


### À vous de jouer

Créez un tool `recherche_catalogue` qui cherche un produit dans un mini catalogue local.

In [ ]:
CATALOGUE = {
    "laptop":  {"nom": "Laptop Pro 15", "prix": 12500, "stock": 8},
    "souris":  {"nom": "Souris sans fil", "prix": 250, "stock": 42},
    "clavier": {"nom": "Clavier mécanique", "prix": 900, "stock": 0},
    "ecran":   {"nom": "Écran 27 pouces", "prix": 3200, "stock": 5},
}

@tool
def recherche_catalogue(mot_cle: str) -> str:
    """Recherche un produit dans le catalogue et retourne son prix (MAD) et son stock."""
    mot = mot_cle.lower()
    for cle, produit in CATALOGUE.items():
        if mot in cle or mot in produit["nom"].lower():
            if produit["stock"] == 0:
                stock_info = "RUPTURE DE STOCK"
            else:
                stock_info = f"Stock : {produit['stock']} unité(s)"
            return f"{produit['nom']} — Prix : {produit['prix']} MAD — {stock_info}"
    return f"Aucun produit trouvé pour '{mot_cle}'"


In [ ]:
# Zone de test — Exercice 1.3
print(recherche_catalogue.invoke({"mot_cle": "souris"}))
print(recherche_catalogue.invoke({"mot_cle": "clavier"}))
print(recherche_catalogue.invoke({"mot_cle": "tablette"}))

## Exercice 1.4 — Tools de recherche web réels avec **Tavily**

Jusqu'ici, toutes nos données (taux de change, catalogue) étaient **statiques,
codées en dur**. Un vrai agent a souvent besoin d'aller chercher une information
**à jour** sur internet : c'est le rôle d'un moteur de recherche comme **Tavily**,
conçu spécifiquement pour être consommé par des agents LLM.

### Documentation — `langchain_tavily.TavilySearch`

```python
from langchain_tavily import TavilySearch
```

`TavilySearch` est un tool **préconstruit** (pas besoin de l'écrire !) qui
interroge l'API de recherche de Tavily et retourne des résultats web structurés
(titre, url, contenu, score de pertinence...).

```python
tavily = TavilySearch(max_results=3, topic="general")
tavily.invoke({"query": "taux de change EUR vers MAD aujourd'hui"})
```

Paramètres utiles à l'instanciation :

| Paramètre | Rôle |
|---|---|
| `max_results` | nombre de résultats retournés |
| `topic` | `"general"` ou `"news"` (actualités) |
| `search_depth` | `"basic"` (rapide) ou `"advanced"` (plus complet) |
| `include_answer` | si `True`, Tavily résume lui-même la réponse |

Il faut une clé **`TAVILY_API_KEY`** dans votre `.env` (voir Partie 0).

### Deux usages différents, deux tools distincts

Plutôt que de donner un seul tool générique à l'agent, on va créer **deux tools
spécialisés**, chacun avec une description précise — cela aide le LLM à choisir
le bon outil au bon moment :

1. **`recherche_taux_change`** : pour obtenir un **taux de change réel et actuel**
   entre deux devises (vient compléter/remplacer `convertir_devise` qui utilise
   des taux figés)
2. **`recherche_web`** : pour une **recherche internet générale** (actualité,
   fait quelconque, définition récente...)

### À vous de jouer

Créez les deux tools ci-dessous, tous les deux basés sur `TavilySearch`, mais
avec un **nom** et une **description** différents pour bien les distinguer.

In [ ]:
from langchain_tavily import TavilySearch

# Instance Tavily partagée, réutilisée par les deux tools ci-dessous
_tavily = TavilySearch(max_results=3, topic="general", search_depth="basic")

@tool
def recherche_web(question: str) -> str:
    """Effectue une recherche internet générale pour répondre à une question
    d'actualité ou une information que tu ne connais pas avec certitude."""
    resultats = _tavily.invoke({"query": question})
    if not resultats:
        return "Aucun résultat trouvé."
    resume = []
    for r in resultats[:3]:
        titre = r.get("title", "Sans titre")
        contenu = r.get("content", "")[:200]
        url = r.get("url", "")
        resume.append(f"• {titre}\n  {contenu}...\n  Source : {url}")
    return "\n\n".join(resume)

@tool
def recherche_taux_change(devise_source: str, devise_cible: str) -> str:
    """Obtient le taux de change actuel et réel entre deux devises via une recherche internet.
    Utilise ce tool pour des taux à jour, contrairement à convertir_devise qui utilise des taux figés."""
    query = f"taux de change {devise_source} vers {devise_cible} aujourd'hui"
    resultats = _tavily.invoke({"query": query})
    if not resultats:
        return f"Impossible d'obtenir le taux de change {devise_source}/{devise_cible}."
    resume = []
    for r in resultats[:2]:
        titre = r.get("title", "Sans titre")
        contenu = r.get("content", "")[:300]
        url = r.get("url", "")
        resume.append(f"• {titre}\n  {contenu}...\n  Source : {url}")
    return "\n\n".join(resume)


In [ ]:
# Zone de test — Exercice 1.4
print("=== Test recherche_taux_change ===")
print(recherche_taux_change.invoke({"devise_source": "EUR", "devise_cible": "MAD"}))

print("\n=== Test recherche_web ===")
print(recherche_web.invoke({"question": "Quelles sont les dernières avancées en intelligence artificielle en 2026 ?"}))


---
# Partie 2 — La boucle ReAct à la main (30 min)

Avant d'utiliser LangGraph, comprenons ce qui se passe **sous le capot**.

Le pattern **ReAct** = **Re**ason + **Act** :

```
1. 🧠 Reason  : le LLM lit la question et RAISONNE → décide d'appeler un tool
2. 🔧 Act     : NOTRE code exécute le tool demandé
3. 👁️ Observe : le résultat est renvoyé au LLM comme "observation"
4. 🔁 Répéter jusqu'à ce que le LLM produise une réponse finale (sans tool)
```

## Exercice 2.1 — Binder les tools et observer `tool_calls`

### Documentation — `llm.bind_tools(tools)`

`bind_tools(liste_de_tools)` retourne un **nouveau** modèle auquel on a "déclaré" les
tools : à chaque appel, leurs noms/descriptions/schémas sont transmis à l'API du LLM.

**Point fondamental** : le LLM **n'exécute jamais** un tool. Quand il veut en
utiliser un, il répond avec un `AIMessage` dont le champ **`tool_calls`** contient
des *demandes* d'exécution :

```python
reponse.tool_calls
# [{'name': 'recherche_catalogue',
#   'args': {'mot_cle': 'souris'},
#   'id': 'call_abc123', ...}]
```

- `name` → quel tool appeler
- `args` → avec quels arguments (dict prêt pour `.invoke()`)
- `id` → identifiant unique pour relier la demande à son résultat (voir 2.2)

C'est **notre code** qui exécute réellement les fonctions Python.

In [ ]:
tools = [additionner, multiplier, diviser, convertir_devise,
         recherche_catalogue, recherche_web, recherche_taux_change]
llm_with_tools = llm.bind_tools(tools)

reponse = llm_with_tools.invoke("Combien coûtent 3 souris en EUR ?")
print("content    :", repr(reponse.content))
print("tool_calls :", reponse.tool_calls)

## Exercice 2.2 — Écrire la boucle ReAct manuellement

### Documentation — Les messages LangChain

```python
from langchain_core.messages import HumanMessage, ToolMessage
```

Une conversation avec un LLM = une **liste de messages** typés :

| Classe | Rôle |
|---|---|
| `HumanMessage` | message de l'utilisateur |
| `AIMessage` | réponse du LLM (peut contenir des `tool_calls`) |
| `ToolMessage` | **résultat d'un tool** renvoyé au LLM (l'observation) |
| `SystemMessage` | instructions de comportement (utilisé en Partie 3) |

Un `ToolMessage` DOIT référencer la demande d'origine via **`tool_call_id`** :

```python
ToolMessage(content=str(resultat), tool_call_id=tool_call["id"])
```

Sans cet identifiant, l'API refuse le message : le LLM ne saurait pas quelle
observation correspond à quelle demande (surtout s'il a appelé plusieurs tools d'un coup).

### À vous de jouer

Complétez la boucle : tant que le LLM demande des tools, on les exécute et on
renvoie les résultats sous forme de `ToolMessage`.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_par_nom = {t.name: t for t in tools}   # dict {"additionner": <tool>, ...}

def boucle_react(question: str, max_iterations: int = 5):
    messages = [HumanMessage(content=question)]
    for i in range(max_iterations):
        reponse = llm_with_tools.invoke(messages)   # 🧠 Reason
        messages.append(reponse)

        if not reponse.tool_calls:                   # plus d'action → réponse finale
            return reponse.content

        for tool_call in reponse.tool_calls:         # 🔧 Act
            print(f"🔧 Action : {tool_call['name']}({tool_call['args']})")
            outil = tools_par_nom[tool_call["name"]]
            resultat = outil.invoke(tool_call["args"])
            print(f"👁️  Observation : {resultat}")
            messages.append(ToolMessage(             # 👁️ Observe
                content=str(resultat),
                tool_call_id=tool_call["id"]
            ))
    return "Nombre maximum d'itérations atteint."

print("\n💬 Réponse finale :")
print(boucle_react("Combien coûtent 3 souris et 2 écrans au total, en EUR ?"))

### Questions de compréhension

1. Pourquoi `tool_call_id` est-il indispensable ?
2. Pourquoi limiter le nombre d'itérations (`max_iterations`) ?

---
# Partie 3 — L'agent ReAct avec LangGraph

Notre boucle `while` devient un **graphe d'états** :

```
START → assistant → (tool_calls ?) ─oui→ tools ─→ assistant → ...
                          │
                         non
                          ↓
                         END
```

Pourquoi un graphe plutôt qu'une boucle ? Parce qu'un graphe est **extensible**
(ajout de nœuds : validation humaine, résumé, routage multi-agents...) et
**persistable** (mémoire, reprise après crash).

## Exercice 3.1 — Construire le graphe

### Documentation — les briques LangGraph

```python
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
```

**`MessagesState`** — le **schéma d'état** du graphe : un dict avec une seule clé
`messages` (la liste des messages de la conversation). Son *reducer* `add_messages`
fait que chaque nœud **ajoute** ses messages à la liste au lieu de la remplacer.

**`StateGraph(MessagesState)`** — le **constructeur du graphe**. Ses méthodes :
- `add_node("nom", fonction)` → ajoute un nœud ; la fonction reçoit l'état et retourne la mise à jour (`{"messages": [...]}`)
- `add_edge(A, B)` → arête fixe : après A, on va toujours à B
- `add_conditional_edges(A, fonction_de_routage)` → après A, la fonction décide de la destination
- `compile()` → fige le graphe en un objet exécutable (`.invoke()`, `.stream()`)

**`START` / `END`** — nœuds virtuels marquant l'entrée et la sortie du graphe.

**`ToolNode(tools)`** — nœud **préconstruit** qui : lit les `tool_calls` du dernier
message, exécute chaque tool, et ajoute les `ToolMessage` à l'état. C'est exactement
ce que vous avez codé à la main en 2.2 !

**`tools_condition`** — fonction de **routage** préconstruite : si le dernier message
contient des `tool_calls` → route vers le nœud `"tools"` ; sinon → route vers `END`.

**`SystemMessage`** — message d'instructions placé en tête de la conversation pour
définir le rôle et les règles de l'agent (son "prompt système").

### À vous de jouer

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = SystemMessage(content=(
    "Tu es un assistant e-commerce. Tu utilises les tools à ta disposition "
    "pour les calculs, les conversions de devises et la recherche produits. "
    "Pour un taux de change réel/à jour, utilise recherche_taux_change plutôt "
    "que convertir_devise. Pour toute question d'actualité ou d'information "
    "que tu ne connais pas avec certitude, utilise recherche_web. "
    "Ne calcule JAMAIS de tête : utilise toujours un tool."
))

def assistant(state: MessagesState):
    """Nœud LLM : raisonne et décide d'appeler (ou non) un tool."""
    reponse = llm_with_tools.invoke([SYSTEM_PROMPT] + state["messages"])
    return {"messages": [reponse]}

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

agent = builder.compile()

### Documentation — visualiser le graphe

Tout graphe compilé expose `get_graph()` :
- `.draw_ascii()` → schéma texte (nécessite le package `grandalf`)
- `.draw_mermaid_png()` → image PNG (dans un notebook, avec `IPython.display.Image`)

In [ ]:
print(agent.get_graph().draw_ascii())

# Version image :
# from IPython.display import Image
# Image(agent.get_graph().draw_mermaid_png())

## Exercice 3.2 — Tester l'agent

### Documentation — `agent.invoke(...)` et `pretty_print()`

L'entrée d'un graphe `MessagesState` est un dict `{"messages": [...]}`.
La sortie est l'**état final** : la liste complète des messages
(question → demandes de tools → observations → réponse finale).

`message.pretty_print()` affiche joliment chaque message avec son type — parfait
pour suivre le raisonnement de l'agent étape par étape.

In [ ]:
resultat = agent.invoke({"messages": [HumanMessage(
    content="J'ai 500 MAD. Puis-je acheter une souris et un clavier ? "
            "Sinon, combien me manque-t-il en EUR ?"
)]})

for m in resultat["messages"]:
    m.pretty_print()

In [ ]:
# Test 1 : question sans tool (l'agent répond directement)
print("=" * 60)
print("TEST 1 — Sans tool : identité de l'agent")
print("=" * 60)
r1 = agent.invoke({"messages": [HumanMessage(content="Bonjour, qui es-tu ?")]})
for m in r1["messages"]:
    m.pretty_print()

# Test 2 : taux de change actuel (déclenche recherche_taux_change)
print("\n" + "=" * 60)
print("TEST 2 — Avec recherche web : taux de change actuel EUR/MAD")
print("=" * 60)
r2 = agent.invoke({"messages": [HumanMessage(content="Quel est le taux de change actuel EUR/MAD ?")]})
for m in r2["messages"]:
    m.pretty_print()


## Exercice 3.3 — Ajouter de la mémoire

### Documentation — `InMemorySaver` (checkpointer)

```python
from langgraph.checkpoint.memory import InMemorySaver
```

Sans mémoire, chaque `invoke` repart de zéro. Un **checkpointer** sauvegarde
l'état du graphe après chaque étape, indexé par un **`thread_id`** (= identifiant
de conversation) passé dans la config :

```python
agent = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "session-1"}}
agent.invoke({"messages": [...]}, config=config)
```

- Même `thread_id` → l'historique est rechargé, l'agent "se souvient"
- `thread_id` différent → conversation vierge
- `InMemorySaver` stocke en RAM (perdu au redémarrage). En production :
  checkpointer **SQLite** ou **Postgres** (`langgraph-checkpoint-sqlite` / `-postgres`).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Recompilation du graphe avec mémoire persistante
agent_memoire = builder.compile(checkpointer=InMemorySaver())

# Le thread_id identifie la conversation — même ID = même historique
config = {"configurable": {"thread_id": "session-1"}}

print("=" * 60)
print("Question 1 : prix de l'écran")
print("=" * 60)
r1 = agent_memoire.invoke(
    {"messages": [HumanMessage(content="Quel est le prix d'un écran ?")]},
    config=config
)
for m in r1["messages"]:
    m.pretty_print()

print("\n" + "=" * 60)
print("Question 2 : conversion en dollars (grâce à la mémoire)")
print("=" * 60)
r2 = agent_memoire.invoke(
    {"messages": [HumanMessage(content="Et en dollars ?")]},
    config=config
)
# On n'affiche que les nouveaux messages (les derniers 3 : human + ai + éventuellement tool)
for m in r2["messages"][-3:]:
    m.pretty_print()
